In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

path = "./qwen2.5-0.5b"
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float32)
model.eval()

print(type(model).__name__, "|", next(model.parameters()).device)

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:07<00:00, 40.59it/s]


Qwen2ForCausalLM | cpu


In [2]:
prompt = "Explain what a watermark is in simple words."

if tokenizer.chat_template:   # Instruct model -> use chat format
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
else:                         # Base model -> plain text
    text = prompt

inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)

print(tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

A watermark is a small image or text that appears on the surface of a document, print, or other material to identify it and prevent unauthorized use. It's often used for legal purposes such as copyright protection, authentication, or tracking information.

Watermarks can be placed on various materials including:

1. Paper: On books, magazines, newspapers, and even packaging.
2. Prints: On prints like calendars, posters, and postcards.
3. Documents: On documents like invoices, receipts, and


In [3]:
with torch.no_grad():
    logits = model(**inputs).logits
print(logits.shape)                      # [1, seq_len, vocab_size]
next_token = logits[0, -1].argmax()
print("Most likely next token:", tokenizer.decode(next_token))

torch.Size([1, 39, 151936])
Most likely next token: A


In [7]:
KEY = "Gopi_Pradeep"


In [68]:
import hashlib
import math

GAMMA = 0.50
VOCAB_SIZE = model.config.vocab_size
top_k = 15
ENTROPY_THRESHOLD = 0.2
def get_green_list(prev_token):
    h = hashlib.sha256(f"{KEY}-{prev_token}".encode()).hexdigest()
    seed = int(h, 16) % (2**32)
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(VOCAB_SIZE, generator=g)
    return set(perm[: int(GAMMA * VOCAB_SIZE)].tolist())

def check_next_token(text, temperature=0.01):
    ids = tokenizer(text, return_tensors="pt")["input_ids"]
    prev_token = ids[0, -1].item()

    with torch.no_grad():
        logits = model(ids).logits[0, -1]

    green = get_green_list(prev_token)

    # ---- convert green set -> boolean mask ----
    mask = torch.zeros(VOCAB, dtype=torch.bool)
    mask[list(green)] = True
    print("Green tokens:", mask.sum().item(), "of", VOCAB)

    # ---- hard + sample: block red, sample from green ----
    logits = logits.clone()
    #logits[~mask] = float("-inf")                    # red tokens get 0 probability
    probs = torch.softmax(logits / temperature, dim=-1)

    sorted_probs, sorted_ids = torch.sort(probs, descending=True)

    top_probs = sorted_probs[:top_k]
    top_ids   = sorted_ids[:top_k]

    #entropy check 
    # normalise so the top-10 sum to 1
    p = top_probs / top_probs.sum()

    # entropy: H = -Σ p·log(p)
    entropy = -(p * torch.log(p + 1e-12)).sum().item()

    # normalised entropy (0 = certain, 1 = completely unsure)
    norm_entropy = entropy / math.log(top_k)

    if norm_entropy < ENTROPY_THRESHOLD:
        # LOW entropy → model is confident → SKIP watermark
        choice = torch.multinomial(p, 1).item()
        next_token = top_ids[choice].item()
        mode = "SKIPPED (low entropy)"
    else:
        wm_logits = logits.clone()
        wm_logits[~mask] = float("-inf")
        wm_probs = torch.softmax(wm_logits / temperature, dim=-1)
        next_token = torch.multinomial(wm_probs, 1).item()
        mode = "WATERMARKED"

     
    next_token = torch.multinomial(probs, 1).item()  # sample one green token
    

    colour = "GREEN" if mask[next_token] else "RED"

    print(f"Previous token : {tokenizer.decode(prev_token)!r} (id {prev_token})")
    print(f"Next token     : {tokenizer.decode(next_token)!r} (id {next_token})")
    print(f"List           : {colour}")

    return tokenizer.decode(next_token)

In [69]:
text = "London is a wonderfull"

for i in range(5):
    next_token = check_next_token(text)
    text = text + next_token

print(text)

Green tokens: 75968 of 151936
Previous token : 'full' (id 8878)
Next token     : ' city' (id 3283)
List           : GREEN
Green tokens: 75968 of 151936
Previous token : ' city' (id 3283)
Next token     : ',' (id 11)
List           : GREEN
Green tokens: 75968 of 151936
Previous token : ',' (id 11)
Next token     : ' but' (id 714)
List           : GREEN
Green tokens: 75968 of 151936
Previous token : ' but' (id 714)
Next token     : ' it' (id 432)
List           : GREEN
Green tokens: 75968 of 151936
Previous token : ' it' (id 432)
Next token     : ' is' (id 374)
List           : GREEN
London is a wonderfull city, but it is


In [67]:
text = "the quick brown"

for i in range(5):
    next_token = check_next_token(text)
    text = text + next_token

print(text)

Green tokens: 75968 of 151936
Previous token : ' brown' (id 13876)
Next token     : ' fox' (id 38835)
List           : RED
Green tokens: 75968 of 151936
Previous token : ' fox' (id 38835)
Next token     : ' jumps' (id 34208)
List           : GREEN
Green tokens: 75968 of 151936
Previous token : ' jumps' (id 34208)
Next token     : ' over' (id 916)
List           : RED
Green tokens: 75968 of 151936
Previous token : ' over' (id 916)
Next token     : ' the' (id 279)
List           : RED
Green tokens: 75968 of 151936
Previous token : ' the' (id 279)
Next token     : ' lazy' (id 15678)
List           : GREEN
the quick brown fox jumps over the lazy


In [16]:
tokenizer("Hello world bro", return_tensors="pt")["input_ids"]

tensor([[9707, 1879, 2896]])